In [ ]:
# IMPORTAÇÃO DAS BIBLIOTECAS PARA O PROJETO
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

In [ ]:
# 1. CARREGAMENTO E UNIFICAÇÃO
file_path = '/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'
xls = pd.ExcelFile(file_path)

df_list = []
for sheet in ['PEDE2022', 'PEDE2023', 'PEDE2024']:
    temp_df = pd.read_excel(xls, sheet_name=sheet)
    temp_df['ANO_REFERENCIA'] = sheet
    df_list.append(temp_df)

df_raw = pd.concat(df_list, ignore_index=True)

In [ ]:
# 2. DEFINIÇÃO DO TARGET
# Criando a lógica de risco baseada no IAN
def definir_risco(ian):
    if str(ian).upper() in ['MODERADA', 'SEVERA']:
        return 1
    return 0
df_raw['TARGET_RISCO'] = df_raw['IAN'].apply(definir_risco)

In [ ]:
# 3. SELEÇÃO DE FEATURES
features_num = ['IDA', 'IEG', 'IPS', 'IPP', 'IAA'] # Indicadores quantitativos
features_cat = ['Fase', 'ANO_REFERENCIA']          # Indicadores qualitativos

X = df_raw[features_num + features_cat]
# Convert the 'Fase' column to string type to avoid mixed-type errors in OneHotEncoder
X['Fase'] = X['Fase'].astype(str)
y = df_raw['TARGET_RISCO']

/tmp/ipython-input-4155630196.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Fase'] = X['Fase'].astype(str)


In [ ]:
# 4. CONSTRUÇÃO DO PIPELINE

# Pipeline para dados numéricos: Trata nulos com a mediana e escala os dados
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para dados categóricos: Trata nulo e converte em colunas binarias
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combinando os processadores
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, features_num),
        ('cat', cat_transformer, features_cat)
    ])

# Pipeline Final: Pré-processamento + Modelo
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

In [ ]:
# 5. TREINO E AVALIAÇÃO
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_pipeline.fit(X_train, y_train)

# Métricas para o Storytelling
y_pred = model_pipeline.predict(X_test)
print("Relatório de Performance:\n", classification_report(y_test, y_pred))

Relatório de Performance:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       606

    accuracy                           1.00       606
   macro avg       1.00      1.00      1.00       606
weighted avg       1.00      1.00      1.00       606



In [ ]:
# 6. EXPORTAÇÃO PARA O STREAMLIT
# Este arquivo 'modelo_passos_magicos.pkl' será usado no Streamlit
joblib.dump(model_pipeline, 'modelo_passos_magicos.pkl')
print("Pipeline exportado com sucesso!")

Pipeline exportado com sucesso!
